# quantum-cq ? teoria, biblioteca e pipeline completa

## Demo para apresenta??o

Este notebook mostra a biblioteca em um formato mais direto para apresenta??o:

- dados de entrada;
- circuito l?gico;
- circuito nativo antes da transpila??o;
- circuito nativo depois da transpila??o;
- execu??o e medi??es;
- perceptron simples processando todos os exemplos em um circuito;
- algoritmo de biblioteca com dados;
- Navigation Encoding V2 com estrutura, lowering, target f?sico/manual e mapeamento l?gico-f?sico.

> A ideia central: manter separadas as camadas **dados ? circuito l?gico ? engine ? target ? execu??o ? resultado**.

In [ ]:
from __future__ import annotations

import math
from pathlib import Path
from pprint import pprint

try:
    from IPython.display import HTML, Markdown, display
except Exception:  # pragma: no cover - fallback fora do Jupyter
    HTML = Markdown = None
    display = print

try:
    import pandas as pd
except Exception:  # pragma: no cover - ambiente m?nimo
    pd = None

from qiskit import QuantumCircuit

from quantum_cq import (
    CQ,
    __version__,
    NativeInstruction,
    StructuralField,
    StructuralHeap,
    StructuralNode,
    StructuralSelector,
    StructuralType,
    TopologyEdge,
)

print('quantum-cq', __version__)
print('reposit?rio:', Path.cwd())

In [ ]:
if HTML is not None:
    display(HTML("""
    <style>
      .cq-hero {
        border-left: 6px solid #276ef1;
        padding: 1rem 1.2rem;
        background: #f6f8fb;
        border-radius: 8px;
        margin: .7rem 0 1rem 0;
      }
      .cq-grid {
        display: grid;
        grid-template-columns: repeat(auto-fit, minmax(210px, 1fr));
        gap: .7rem;
        margin: .7rem 0 1rem 0;
      }
      .cq-card {
        border: 1px solid #d8dee9;
        border-radius: 8px;
        padding: .8rem .9rem;
        background: white;
      }
      .cq-card b { color: #1f3b73; }
      table.dataframe { font-size: 0.92rem; }
    </style>
    <div class="cq-hero">
      <b>Formato de apresenta??o:</b> cada bloco mostra primeiro a inten??o, depois os dados, depois o circuito e por fim o resultado medido.
    </div>
    """))

## 1. Helpers de apresenta??o

As fun??es abaixo apenas inspecionam objetos p?blicos da biblioteca. Elas n?o alteram circuitos nem executam passes escondidos.

In [ ]:
def show_table(rows, title=None):
    if title and Markdown is not None:
        display(Markdown(f'### {title}'))
    if pd is not None:
        display(pd.DataFrame(list(rows)))
    else:
        for row in rows:
            print(row)


def metric_payload(metrics):
    payload = {}
    for key, value in dict(metrics or {}).items():
        payload[key] = getattr(value, 'value', value)
    return payload


def circuit_summary(circuit):
    if hasattr(circuit, 'num_qubits'):
        return {
            'type': type(circuit).__name__,
            'qubits': circuit.num_qubits,
            'clbits': circuit.num_clbits,
            'depth': circuit.depth() if hasattr(circuit, 'depth') else None,
            'size': circuit.size() if hasattr(circuit, 'size') else None,
            'ops': dict(circuit.count_ops()) if hasattr(circuit, 'count_ops') else None,
        }
    return {
        'type': type(circuit).__name__,
        'name': getattr(circuit, 'name', None),
        'qubits': getattr(circuit, 'n_qubits', None),
        'clbits': getattr(circuit, 'n_clbits', None),
        'layers': len(getattr(circuit, 'layers', ())),
        'outputs': len(getattr(circuit, 'outputs', ())),
        'metadata': dict(getattr(circuit, 'metadata', {}) or {}),
    }


def instruction_table(circuit, limit=36):
    if not hasattr(circuit, 'data'):
        rows = []
        for layer_index, layer in enumerate(getattr(circuit, 'layers', ())):
            for operation in getattr(layer, 'operations', ()):
                rows.append({
                    '#': len(rows),
                    'stage': f'layer-{layer_index}',
                    'gate': operation.kind,
                    'qubits': tuple(operation.qubits),
                    'clbits': tuple(operation.clbits),
                    'params': dict(operation.params),
                })
        for operation in getattr(circuit, 'outputs', ()):
            rows.append({
                '#': len(rows),
                'stage': 'outputs',
                'gate': operation.kind,
                'qubits': tuple(operation.qubits),
                'clbits': tuple(operation.clbits),
                'params': dict(operation.params),
            })
        return rows[:limit] + ([{'#': '...', 'stage': '...', 'gate': f'+{len(rows)-limit} instru??es', 'qubits': (), 'clbits': (), 'params': {}}] if len(rows) > limit else [])

    qubit_index = {bit: index for index, bit in enumerate(circuit.qubits)}
    clbit_index = {bit: index for index, bit in enumerate(circuit.clbits)}
    rows = []
    for index, inst in enumerate(circuit.data[:limit]):
        operation = inst.operation
        rows.append({
            '#': index,
            'gate': operation.name,
            'qubits': tuple(qubit_index[q] for q in inst.qubits),
            'clbits': tuple(clbit_index[c] for c in inst.clbits),
            'params': tuple(str(param) for param in operation.params),
        })
    if len(circuit.data) > limit:
        rows.append({'#': '...', 'gate': f'+{len(circuit.data)-limit} instru??es', 'qubits': (), 'clbits': (), 'params': ()})
    return rows


def show_circuit(circuit, title):
    if Markdown is not None:
        display(Markdown(f'### {title}'))
    pprint(circuit_summary(circuit))
    show_table(instruction_table(circuit), 'Instru??es')


def show_pipeline(result, title):
    scenario = result.scenario_results[0]
    if Markdown is not None:
        display(Markdown(f'## {title}'))
    print('scenario_id:', scenario.scenario_id)
    print('status:', scenario.status)
    show_table(
        [
            {
                'stage': stage.stage_id,
                'status': stage.status,
                'requires': ', '.join(stage.requires),
                'provides': ', '.join(stage.provides),
            }
            for stage in scenario.stage_results
        ],
        'Stages',
    )
    show_table(
        [
            {
                'snapshot': snapshot.snapshot_id,
                'stage': snapshot.stage_id,
                'format': snapshot.format,
                'engine': snapshot.engine,
                'target': snapshot.target,
                **metric_payload(snapshot.metrics),
            }
            for snapshot in scenario.snapshots
        ],
        'Snapshots',
    )
    return scenario


def snapshot_by_stage(result, stage_id):
    scenario = result.scenario_results[0]
    matches = [snapshot for snapshot in scenario.snapshots if snapshot.stage_id == stage_id]
    return matches[-1] if matches else None


def dominant_count(counts):
    bitstring = max(counts, key=counts.get)
    return bitstring, counts[bitstring]


def bits(value, width):
    return format(value, f'0{width}b')

# Parte I ? dados e circuitos l?gicos

## 2. Perceptron simples em lote

Vamos usar um perceptron bin?rio que implementa a regra OR:

$$
\hat y = 1 \quad \text{se} \quad w_0 x_0 + w_1 x_1 + b > 0.
$$

A vers?o revers?vel usada no circuito escreve a sa?da em um registrador separado:

$$
y = x_0 \lor x_1 = x_0 \oplus x_1 \oplus (x_0 \land x_1).
$$

Os quatro exemplos s?o processados **em um ?nico circuito**.

In [ ]:
samples = [
    {'sample': 's0', 'x0': 0, 'x1': 0, 'target': 0},
    {'sample': 's1', 'x0': 0, 'x1': 1, 'target': 1},
    {'sample': 's2', 'x0': 1, 'x1': 0, 'target': 1},
    {'sample': 's3', 'x0': 1, 'x1': 1, 'target': 1},
]

def train_perceptron(rows, epochs=4, learning_rate=1):
    weights = [0, 0]
    bias = 0
    history = []
    for epoch in range(epochs):
        errors = 0
        for row in rows:
            score = weights[0] * row['x0'] + weights[1] * row['x1'] + bias
            prediction = int(score > 0)
            error = row['target'] - prediction
            weights[0] += learning_rate * error * row['x0']
            weights[1] += learning_rate * error * row['x1']
            bias += learning_rate * error
            errors += abs(error)
        history.append({'epoch': epoch + 1, 'weights': tuple(weights), 'bias': bias, 'errors': errors})
    return tuple(weights), bias, history

weights, bias, training_history = train_perceptron(samples)
for row in samples:
    row['score'] = weights[0] * row['x0'] + weights[1] * row['x1'] + bias
    row['prediction'] = int(row['score'] > 0)

show_table(training_history, 'Treinamento cl?ssico m?nimo')
show_table(samples, 'Dados usados pelo perceptron')
print('pesos finais:', weights, 'bias:', bias)

## 3. Um ?nico circuito para os quatro exemplos

Cada exemplo recebe tr?s qubits l?gicos:

- `x0`;
- `x1`;
- `y`, inicialmente em `0`.

A sa?da medida em `y` ? a predi??o daquele exemplo.

In [ ]:
def build_batch_perceptron_or(rows):
    builder = CQ.circuit(
        qubits=3 * len(rows),
        clbits=len(rows),
        name='perceptron_or_batch',
        metadata={'demo': 'batch reversible OR perceptron', 'weights': weights, 'bias': bias},
    )
    layout_rows = []
    for index, row in enumerate(rows):
        q0, q1, y = 3 * index, 3 * index + 1, 3 * index + 2
        if row['x0']:
            builder.x(q0)
        if row['x1']:
            builder.x(q1)
        builder.cx(q0, y)
        builder.cx(q1, y)
        builder.ccx(q0, q1, y)
        builder.measure(y, index)
        layout_rows.append({
            'sample': row['sample'],
            'x0': row['x0'],
            'x1': row['x1'],
            'target': row['target'],
            'q_x0': q0,
            'q_x1': q1,
            'q_y': y,
            'clbit_prediction': index,
        })
    return builder.build(), layout_rows

perceptron_ir, perceptron_layout = build_batch_perceptron_or(samples)
show_table(perceptron_layout, 'Mapeamento dados -> qubits -> clbits')
show_circuit(perceptron_ir, 'Circuito l?gico do perceptron em lote')

# Parte II ? pipeline, transpila??o e medi??es

## 4. Pipeline do perceptron

Agora o mesmo circuito passa pela pipeline:

1. entrada l?gica `CircuitIR`;
2. emiss?o nativa Qiskit;
3. transpila??o nativa;
4. compila??o;
5. execu??o;
6. normaliza??o dos counts.

In [ ]:
perceptron_transpiled = CQ.pipeline(circuit=perceptron_ir, engine='qiskit').transpile()
scenario = show_pipeline(perceptron_transpiled, 'Pipeline do perceptron: antes e depois da transpila??o')

logical_snapshot = snapshot_by_stage(perceptron_transpiled, 'input_adapt')
native_before = snapshot_by_stage(perceptron_transpiled, 'native_emit')
native_after = snapshot_by_stage(perceptron_transpiled, 'native_transpile')

show_circuit(logical_snapshot.circuit, 'Antes da transpila??o: circuito l?gico')
show_circuit(native_before.circuit, 'Antes da transpila??o nativa: Qiskit emitido')
show_circuit(native_after.circuit, 'Depois da transpila??o nativa')

print('record de transpila??o:')
pprint(scenario.artifacts.get('transpilation_record'))

In [ ]:
try:
    perceptron_executed = CQ.pipeline(circuit=perceptron_ir, engine='qiskit').run_engine(engine='qiskit', shots=128)
    counts = perceptron_executed.engine_result.counts
    bitstring, count = dominant_count(counts)
    # Counts can?nicos: clbit de maior ?ndice ? esquerda. Para voltar para sample s0..s3, invertemos.
    predicted_by_sample = [int(bit) for bit in bitstring[::-1]]
    measurement_rows = []
    for row, measured in zip(samples, predicted_by_sample):
        measurement_rows.append({
            'sample': row['sample'],
            'x0': row['x0'],
            'x1': row['x1'],
            'target': row['target'],
            'classical_prediction': row['prediction'],
            'measured_prediction': measured,
        })
    show_table([{'bitstring_canonical': key, 'count': value} for key, value in counts.items()], 'Counts can?nicos')
    show_table(measurement_rows, 'Predi??es recuperadas por medi??o')
    assert [row['target'] for row in samples] == predicted_by_sample
except Exception as exc:
    perceptron_executed = None
    print('Execu??o local indispon?vel:', type(exc).__name__, exc)

## 5. Algoritmo pronto usando dados: Bernstein-Vazirani

Aqui a entrada de dados ? o segredo `1011`. A biblioteca monta o algoritmo, a pipeline executa e os counts medidos recuperam o segredo considerando a conven??o can?nica de bits.

In [ ]:
secret = '1011'
bv_algorithm = CQ.bv(secret)
show_circuit(CQ.to_qiskit(bv_algorithm), 'Bernstein-Vazirani gerado pela biblioteca')
print('metadata do algoritmo:')
pprint(bv_algorithm.metadata)

try:
    bv_result = CQ.pipeline(circuit=bv_algorithm, engine='qiskit').run_engine(engine='qiskit', shots=64)
    bv_counts = bv_result.engine_result.counts
    dominant, _ = dominant_count(bv_counts)
    recovered_secret = dominant[::-1]
    show_table([{'canonical_count': key, 'shots': value} for key, value in bv_counts.items()], 'Medi??es BV')
    print('secret esperado:', secret)
    print('secret recuperado:', recovered_secret)
    assert recovered_secret == secret
except Exception as exc:
    bv_result = None
    print('Execu??o local indispon?vel:', type(exc).__name__, exc)

# Parte III ? Navigation Encoding V2

## 6. Estrutura tipada: lista ligada finita

A V2 trabalha sobre estrutura, n?o sobre endere?o f?sico. Nesta demo:

- cada n? tem `payload` de 4 bits;
- cada n? tem uma refer?ncia `link` com papel sem?ntico `next`;
- a opera??o V2 ? `read(payload)`;
- o lowering exato reutiliza V1/XOR-load por baixo, mas preserva metadata estrutural.

In [ ]:
values = [3, 5, 7]
node_type = StructuralType(
    'Node',
    (
        StructuralField('payload', 'uint', bit_width=4, semantic_role='value'),
        StructuralField('link', 'reference', nullable=True, semantic_role='next'),
    ),
)
heap = StructuralHeap(
    (node_type,),
    (
        StructuralNode('node0', 'Node', {'payload': values[0], 'link': 'node1'}),
        StructuralNode('node1', 'Node', {'payload': values[1], 'link': 'node2'}),
        StructuralNode('node2', 'Node', {'payload': values[2], 'link': None}),
    ),
    roots=('node0',),
)

nav_v2 = CQ.navigation_v2(
    heap,
    operation='read',
    selector=StructuralSelector.value('payload'),
    metadata={'demo': 'presentation navigation v2 read'},
)

print('navigation_version:', nav_v2.navigation_version)
print('operation:', nav_v2.operation.operation)
print('circuit_format:', nav_v2.circuit_format)
print('equivalence_fingerprint:', nav_v2.equivalence_class.equivalence_fingerprint)
print('lowering_backend:', nav_v2.metadata.get('lowering_backend'))
show_table(
    [
        {'local_node_id': local, 'canonical_node_id': canonical}
        for local, canonical in nav_v2.equivalence_class.local_to_canonical.items()
    ],
    'Renomea??o local -> can?nica',
)
show_table(
    [
        {'canonical_node_id': node, 'encoded_pointer_value': code, 'bits': bits(code, nav_v2.plan.pointer_width)}
        for node, code in nav_v2.plan.register_layout['pointer']['codes'].items()
    ] + [{'canonical_node_id': 'NULL', 'encoded_pointer_value': nav_v2.plan.null_encoding, 'bits': bits(nav_v2.plan.null_encoding, nav_v2.plan.pointer_width)}],
    'C?digos finitos de ponteiro',
)
print('memory/rho table:', nav_v2.plan.memory_values)

## 7. Como ficam os qubits l?gicos da Navigation V2

A estrutura V2 ainda n?o ? hardware. Ela gera um plano finito:

- registrador de ponteiro;
- seletor, quando necess?rio;
- registrador acumulador de sa?da;
- depois do placement, esses qubits l?gicos s?o associados a qubits f?sicos do target.

In [ ]:
target_qubits = tuple(f'p{i}' for i in range(nav_v2.circuit.n_qubits))
line_edges = tuple(
    TopologyEdge(target_qubits[i], target_qubits[i + 1], directed=False, operations=('cx', 'swap'))
    for i in range(len(target_qubits) - 1)
)
nav_target = CQ.manual_target(
    target_id='presentation_line_nav_v2',
    qubits=target_qubits,
    operations=(
        NativeInstruction('x', arity=1),
        NativeInstruction('cx', arity=2),
        NativeInstruction('ccx', arity=3),
        NativeInstruction('mcx', arity=4),
        NativeInstruction('measure', arity=1),
    ),
    topology=line_edges,
    target_type='simulator_ideal',
)

nav_planned = CQ.pipeline(
    structural_navigation=nav_v2,
    target=nav_target,
    placement='identity',
    stages=(
        'navigation_v2_validate',
        'navigation_v2_canonicalize',
        'navigation_v2_plan',
        'navigation_v2_lower',
        'navigation_v2_verify',
        'placement',
    ),
    stop_after='placement',
).transpile()
nav_scenario = show_pipeline(nav_planned, 'Navigation V2 at? placement f?sico')
placement = nav_scenario.artifacts['placement_plan']

pointer_width = nav_v2.plan.pointer_width
output_width = nav_v2.plan.output_width
layout_rows = []
for logical in range(nav_v2.circuit.n_qubits):
    if logical < pointer_width:
        register = 'pointer'
        meaning = f'bit {logical} do ponteiro can?nico'
    else:
        register = 'output'
        meaning = f'bit {logical - pointer_width} do payload lido'
    layout_rows.append({
        'logical_qubit': logical,
        'register': register,
        'meaning': meaning,
        'physical_qubit_after_placement': placement.logical_to_physical[logical],
    })
show_table(layout_rows, 'Qubits l?gicos V2 -> qubits f?sicos do target')
show_table(
    [
        {'edge': f'{edge.source} -- {edge.target}', 'directed': edge.directed, 'operations': ', '.join(edge.operations)}
        for edge in nav_target.architecture.topology
    ],
    'Topologia f?sica declarada',
)

In [ ]:
def draw_physical_target(target):
    try:
        import matplotlib.pyplot as plt
    except Exception as exc:
        print('matplotlib indispon?vel:', type(exc).__name__, exc)
        return
    qubits = list(target.architecture.qubits)
    positions = {qubit: (index, 0) for index, qubit in enumerate(qubits)}
    fig, ax = plt.subplots(figsize=(max(7, len(qubits) * 1.1), 2.2))
    for edge in target.architecture.topology:
        x0, y0 = positions[edge.source]
        x1, y1 = positions[edge.target]
        ax.plot([x0, x1], [y0, y1], color='#778899', linewidth=2)
        ax.text((x0 + x1) / 2, 0.11, ','.join(edge.operations), ha='center', fontsize=9)
    for qubit, (x, y) in positions.items():
        ax.scatter([x], [y], s=900, color='#276ef1', edgecolor='black', zorder=3)
        ax.text(x, y, qubit, color='white', ha='center', va='center', fontweight='bold')
    ax.set_title(f'Target {target.descriptor.target_id}: topologia f?sica declarada')
    ax.set_axis_off()
    display(fig)
    plt.close(fig)

draw_physical_target(nav_target)

## 8. Navegando e recuperando dados com V2

Agora preparamos o ponteiro para `node2`, aplicamos o operador V2 baixado e medimos o payload. O valor esperado ? `7`.

In [ ]:
def build_navigation_v2_query(nav_result, local_node_id):
    canonical_id = nav_result.equivalence_class.local_to_canonical[local_node_id]
    pointer_code = nav_result.plan.register_layout['pointer']['codes'][canonical_id]
    pointer_width = nav_result.plan.pointer_width
    output_width = nav_result.plan.output_width
    operator_qc = CQ.to_qiskit(nav_result.circuit)

    query = QuantumCircuit(pointer_width + output_width, output_width, name=f'v2_read_{local_node_id}')
    for bit in range(pointer_width):
        if (pointer_code >> bit) & 1:
            query.x(bit)
    query.compose(operator_qc, inplace=True)
    for bit in range(output_width):
        query.measure(pointer_width + bit, bit)
    return query, canonical_id, pointer_code

nav_query, canonical_node, pointer_code = build_navigation_v2_query(nav_v2, 'node2')
print('local node:', 'node2')
print('canonical node:', canonical_node)
print('encoded pointer value:', pointer_code, 'bits:', bits(pointer_code, nav_v2.plan.pointer_width))
show_circuit(nav_query, 'Circuito de consulta V2 para node2')

nav_query_pipeline = CQ.pipeline(circuit=nav_query, engine='qiskit').transpile()
show_pipeline(nav_query_pipeline, 'Consulta V2: circuito antes/depois da transpila??o')
show_circuit(snapshot_by_stage(nav_query_pipeline, 'native_emit').circuit, 'Consulta V2 antes da transpila??o nativa')
show_circuit(snapshot_by_stage(nav_query_pipeline, 'native_transpile').circuit, 'Consulta V2 depois da transpila??o nativa')

try:
    nav_query_result = CQ.pipeline(circuit=nav_query, engine='qiskit').run_engine(engine='qiskit', shots=128)
    nav_counts = nav_query_result.engine_result.counts
    bitstring, count = dominant_count(nav_counts)
    recovered_value = int(bitstring, 2)
    show_table([{'canonical_count': key, 'shots': value} for key, value in nav_counts.items()], 'Medi??es Navigation V2')
    print('valor esperado:', values[2])
    print('valor recuperado:', recovered_value)
    assert recovered_value == values[2]
except Exception as exc:
    nav_query_result = None
    print('Execu??o local indispon?vel:', type(exc).__name__, exc)

# Fechamento

## O que este notebook demonstrou

<div class="cq-grid">
  <div class="cq-card"><b>Dados</b><br>Entradas cl?ssicas usadas explicitamente no perceptron e no BV.</div>
  <div class="cq-card"><b>Circuitos</b><br>CircuitIR l?gico, Qiskit emitido e Qiskit transpilado.</div>
  <div class="cq-card"><b>Medi??es</b><br>Counts can?nicos e decodifica??o por conven??o de bits.</div>
  <div class="cq-card"><b>Navigation V2</b><br>Heap tipada, forma can?nica, ponteiros, lowering e qubits l?gico-f?sicos.</div>
</div>

A regra pr?tica para apresenta??o ?:

1. mostre a entrada;
2. mostre o circuito l?gico;
3. mostre o circuito nativo antes/depois da transpila??o;
4. mostre as medi??es;
5. explique o mapeamento f?sico somente depois do lowering l?gico.

In [ ]:
summary = {
    'perceptron_counts': None if perceptron_executed is None else dict(perceptron_executed.engine_result.counts),
    'bv_counts': None if bv_result is None else dict(bv_result.engine_result.counts),
    'navigation_v2_counts': None if nav_query_result is None else dict(nav_query_result.engine_result.counts),
    'navigation_v2_pointer_width': nav_v2.plan.pointer_width,
    'navigation_v2_output_width': nav_v2.plan.output_width,
    'target_qubits': nav_target.architecture.qubits,
    'placement': dict(placement.logical_to_physical),
}
summary